# Program 10 – MLP Weight Initialization Methods

**Objective:** Test how weight initialization affects training.

**Dataset:** MNIST (switch to Fashion-MNIST by changing one line)

**Key tasks**
1. Build 3 models using different initializers: `he_normal`, `glorot_uniform`, `random_normal`.
2. Train and compare convergence rate.

## Theory (for viva)

Initial weights decide how signals and gradients flow at the start of training.
- Too **small** → activations and gradients shrink layer by layer (**vanishing**).
- Too **large** → they grow layer by layer (**exploding**) / neurons saturate.
- All **zeros** → every neuron in a layer learns the same thing (symmetry is never broken).

| Initializer | Distribution | Variance | Best for |
|---|---|---|---|
| `random_normal` (Keras default) | $\mathcal{N}(0, 0.05^2)$ | fixed 0.0025 — ignores layer size | nothing in particular (baseline) |
| `glorot_uniform` (Xavier) | $U[-l, l],\ l=\sqrt{\frac{6}{n_{in}+n_{out}}}$ | $\frac{2}{n_{in}+n_{out}}$ | tanh / sigmoid |
| `he_normal` (Kaiming) | truncated $\mathcal{N}(0, \frac{2}{n_{in}})$ | $\frac{2}{n_{in}}$ | ReLU (ReLU zeroes half the inputs, so variance is doubled) |

Glorot and He **scale variance with layer size** so that activation variance stays roughly constant through depth.
To make the effect visible we use a **deep** (6 hidden layers) ReLU network with plain **SGD** (Adam would hide much of the difference).

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

## Step 2 – Load data

In [ ]:
dataset = keras.datasets.mnist                                      # Change to keras.datasets.fashion_mnist if needed
(x_train, y_train), (x_test, y_test) = dataset.load_data()          # Load images and labels
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0        # Flatten + normalize
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0          # Flatten + normalize
print(x_train.shape, x_test.shape)                                  # (60000, 784) (10000, 784)

## Step 3 – Model builder that takes an initializer

In [ ]:
HIDDEN_LAYERS = [256, 256, 128, 128, 64, 64]                        # 6 hidden layers (deep enough to show the effect)


def build_model(initializer):                                       # initializer: string name or Initializer object
    model = keras.Sequential(name=f"mlp_{initializer}")             # Named model
    model.add(keras.Input(shape=(784,)))                            # Input layer
    for units in HIDDEN_LAYERS:                                     # Add each hidden layer
        model.add(layers.Dense(units, activation="relu",            # Dense + ReLU
                               kernel_initializer=initializer))     # <-- the only thing we change
    model.add(layers.Dense(10, activation="softmax",                # Output layer
                           kernel_initializer=initializer))         # Same initializer for output
    return model                                                    # Return model

initializers = ["he_normal", "glorot_uniform", "random_normal"]     # The three initializers to compare

## Step 4 – Look at initial weights and activations BEFORE training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))                     # One histogram per initializer
for ax, init in zip(axes, initializers):                            # Loop over initializers
    tf.random.set_seed(0)                                           # Same random seed
    m = build_model(init)                                           # Untrained model
    w = m.layers[0].get_weights()[0].flatten()                      # First-layer weights as 1-D array
    ax.hist(w, bins=100, color="tab:blue")                          # Histogram of weight values
    ax.set_title(f"{init}\nstd = {w.std():.4f}")                    # Show standard deviation
    ax.set_xlabel("weight value")                                   # X label
plt.suptitle("First-layer initial weight distributions")            # Title
plt.tight_layout()                                                  # Tidy
plt.show()                                                          # Show

In [ ]:
sample = x_train[:1000]                                             # 1000 images to push through the untrained nets

plt.figure(figsize=(9, 5))                                          # Figure
for init in initializers:                                           # Loop over initializers
    tf.random.set_seed(0)                                           # Same seed
    m = build_model(init)                                           # Untrained model
    stds = []                                                       # Std of activations per layer
    a = sample                                                      # Start with input
    for layer in m.layers[:-1]:                                     # Hidden layers only
        a = layer(a)                                                # Forward through one layer
        stds.append(float(tf.math.reduce_std(a)))                   # Record activation spread
    plt.plot(range(1, len(stds) + 1), stds, "o-", label=init)       # Plot std vs layer depth
plt.yscale("log")                                                   # Log scale shows shrinking clearly
plt.xlabel("Hidden layer")                                          # X label
plt.ylabel("Activation std (log)")                                  # Y label
plt.title("Activation spread through depth at initialization")      # Title
plt.legend()                                                        # Legend
plt.grid(True)                                                      # Grid
plt.show()                                                          # random_normal activations shrink with depth

## Step 5 – Train each model with identical settings

In [ ]:
EPOCHS = 15                                                         # Epochs per model
histories = {}                                                      # Store histories
test_acc = {}                                                       # Store test accuracies

for init in initializers:                                           # Loop over initializers
    print(f"\n===== {init} =====")                                  # Header
    tf.random.set_seed(42)                                          # Same seed for all runs (only init type differs)
    model = build_model(init)                                       # Build model
    model.compile(                                                  # Compile
        optimizer=keras.optimizers.SGD(learning_rate=0.01),         # Plain SGD shows init effect clearly
        loss="sparse_categorical_crossentropy",                     # Integer labels
        metrics=["accuracy"],                                       # Accuracy
    )
    h = model.fit(x_train, y_train,                                 # Train
                  epochs=EPOCHS, batch_size=128,                    # Epochs and batch size
                  validation_split=0.1, verbose=2)                  # 10% validation, 1 line/epoch
    histories[init] = h.history                                     # Save history
    test_acc[init] = model.evaluate(x_test, y_test, verbose=0)[1]   # Save test accuracy

## Step 6 – Compare convergence curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))                     # Three plots
ep = np.arange(1, EPOCHS + 1)                                       # X-axis
for init, h in histories.items():                                   # One line per initializer
    axes[0].plot(ep, h["loss"], "o-", label=init)                   # Training loss
    axes[1].plot(ep, h["val_loss"], "o-", label=init)               # Validation loss
    axes[2].plot(ep, h["val_accuracy"], "o-", label=init)           # Validation accuracy
for ax, t in zip(axes, ["Training loss", "Validation loss", "Validation accuracy"]):  # Decorate plots
    ax.set_title(t)                                                 # Title
    ax.set_xlabel("Epoch")                                          # X label
    ax.legend()                                                     # Legend
    ax.grid(True)                                                   # Grid
plt.suptitle("Effect of weight initialization (deep ReLU MLP, SGD)")  # Overall title
plt.tight_layout()                                                  # Tidy
plt.show()                                                          # Show

## Step 7 – Convergence-rate table

In [ ]:
TARGET = 0.90                                                       # Accuracy threshold to measure "speed"

print(f"{'Initializer':<16} {'Epoch-1 loss':>12} {'Final loss':>11} "
      f"{'Epochs to 90% val':>18} {'Test acc':>9}")                 # Table header
print("-" * 70)                                                     # Separator
for init, h in histories.items():                                   # One row per initializer
    reached = [i + 1 for i, a in enumerate(h["val_accuracy"]) if a >= TARGET]  # Epochs where val acc >= 90%
    first = reached[0] if reached else "never"                      # First such epoch, or 'never'
    print(f"{init:<16} {h['loss'][0]:>12.4f} {h['loss'][-1]:>11.4f} "
          f"{str(first):>18} {test_acc[init]:>9.4f}")               # Print row

## Expected observations
- **he_normal** converges fastest with ReLU (lowest loss after epoch 1).
- **glorot_uniform** is close behind (slightly slower for ReLU).
- **random_normal** (fixed std 0.05) makes activations shrink layer after layer → loss stays near $\ln 10 \approx 2.30$ for several epochs (vanishing signal) and converges much later.

## Viva questions
1. Why not initialize all weights to zero? → All neurons compute the same output and get the same gradient → symmetry never breaks.
2. Why does He use $2/n_{in}$ instead of $1/n_{in}$? → ReLU zeroes about half the activations, halving variance; factor 2 compensates.
3. Which initializer for tanh/sigmoid? → Glorot/Xavier.
4. What is Keras's default kernel initializer for Dense? → `glorot_uniform`.
5. Why use SGD instead of Adam here? → Adam adapts step sizes per weight and partly hides bad initialization.